# 輝度ムラ解析・評価 統合ワークフロー

このノートブックは以下の手順で処理を行います。
1. カメラで撮影したRaw画像(.tiff)からGチャンネルを抽出し、輝度マップへ変換（対象画像・ビネット画像）
2. ビネット画像を用いて対象画像の周辺減光を補正
3. 補正画像を保存（ここで手動クリッピングを行う）
4. クリップ済み画像を読み込み、正規化して輝度ムラを可視化（グレースケール & Inferno）
5. 元のRaw画像上の特定ROIにおけるRGB統計量を計算

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# --- 設定パラメータ ---
# 作業ディレクトリ（適宜変更してください）
BASE_DIR = r"C:\Users\HamaKazu\Desktop\GradSchool\lab\experiment\DisplayBrightness\pattern\BG_brightness_pattern"

# 入力ファイル名
TARGET_FILENAME = "bg.tiff"       # 処理対象（ディスク撮影画像）
REF_FILENAME = "bn.tiff"          # ビネット補正用（白画像）

# 出力ファイル名
CORRECTED_FILENAME = "corrected_full.png"  # 補正後（クリップ前）
CLIPPED_FILENAME = "corrected_clipped.png" # 手動クリップ後の読み込み用

# パスの結合
target_path = os.path.join(BASE_DIR, TARGET_FILENAME)
ref_path = os.path.join(BASE_DIR, REF_FILENAME)
corrected_path = os.path.join(BASE_DIR, CORRECTED_FILENAME)
clipped_path = os.path.join(BASE_DIR, CLIPPED_FILENAME)

print(f"Target: {target_path}")
print(f"Reference: {ref_path}")

## 1. Raw画像読み込み & Gチャンネル抽出
Bayer BG8形式のTIFF画像を読み込み、デモザイク処理を行ってGチャンネルのみを抽出します。

In [ ]:
def load_and_extract_green(path):
    """Raw画像を読み込み、BayerBG -> RGB変換後、Gチャンネルを返す"""
    # 16bit/8bitを維持して読み込み
    raw = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise FileNotFoundError(f"画像が見つかりません: {path}")
    
    # デモザイク処理 (Bayer BG パターンと仮定)
    # cv2.COLOR_BayerBG2RGB を使用
    rgb = cv2.cvtColor(raw, cv2.COLOR_BayerBG2RGB)
    
    # Gチャンネル抽出 (OpenCVはRGB変換後、並びはRGBになるはずだが念のため確認)
    # cv2.cvtColorの戻り値はRGB指定ならRGB、BGR指定ならBGR。
    # ここでは COLOR_BayerBG2RGB なので [R, G, B] の順。Gはインデックス1。
    g_channel = rgb[:, :, 1]
    
    return raw, rgb, g_channel

# 画像の読み込み処理
try:
    raw_target, rgb_target, g_target = load_and_extract_green(target_path)
    raw_ref, rgb_ref, g_ref = load_and_extract_green(ref_path)
    print("画像の読み込みとGチャンネル抽出が完了しました。")
    print(f"Target Shape: {g_target.shape}, Max: {np.max(g_target)}")
    print(f"Ref Shape: {g_ref.shape}, Max: {np.max(g_ref)}")
except Exception as e:
    print(e)
    # エラー時は停止
    raise e

## 2. 周辺減光（ビネット）補正
リファレンス画像からゲインマップを作成し、ターゲット画像を補正します。

In [ ]:
# float型に変換して計算
target_f = g_target.astype(np.float32)
ref_f = g_ref.astype(np.float32)

# ゲインマップ作成 (Ref / Max(Ref))
ref_max = np.max(ref_f)
gain_map = ref_f / ref_max

# 0除算回避
gain_map = np.maximum(gain_map, 1e-6)

# 補正実行 (Target / GainMap)
corrected_f = target_f / gain_map

# クリッピングと型変換 (元のビット深度に合わせる)
if g_target.dtype == np.uint16:
    max_val = 65535
    out_dtype = np.uint16
else:
    max_val = 255
    out_dtype = np.uint8

corrected_img = np.clip(corrected_f, 0, max_val).astype(out_dtype)

## 3. 補正画像の保存と手動クリッピング
ここで一度画像を保存します。**手動で画像を開き、ディスプレイ部分をトリミング（クリッピング）して上書き保存、または別名保存してください。**

In [ ]:
# 保存
cv2.imwrite(corrected_path, corrected_img)
print(f"補正画像を保存しました: {corrected_path}")
print("-" * 60)
print("【重要】")
print(f"1. 上記のファイル '{os.path.basename(corrected_path)}' を画像編集ソフトで開いてください。")
print(f"2. 解析したいディスプレイ領域のみを切り抜き（トリミング）してください。")
print(f"3. 切り抜いた画像を '{os.path.basename(clipped_path)}' という名前で保存してください。")
print("-" * 60)

## 4. 正規化と輝度ムラの可視化
クリップされた画像を読み込み、Min-Max正規化を行って表示します。

In [ ]:
if not os.path.exists(clipped_path):
    print(f"エラー: クリップ後の画像が見つかりません: {clipped_path}")
    # デモ用に補正画像をそのまま使う場合のフォールバック（本番ではコメントアウト推奨）
    # clipped_img_raw = corrected_img
else:
    clipped_img_raw = cv2.imread(clipped_path, cv2.IMREAD_UNCHANGED)
    print(f"クリップ画像を読み込みました: {clipped_path}")

    # 正規化処理
    img_float = clipped_img_raw.astype(np.float32)
    v_min = np.min(img_float)
    v_max = np.max(img_float)

    if v_max - v_min == 0:
        img_norm = np.zeros_like(img_float)
    else:
        img_norm = (img_float - v_min) / (v_max - v_min)

    # 可視化
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. 0-1 グレースケール
    im1 = axes[0].imshow(img_norm, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f"Normalized Grayscale (0.0 - 1.0)\nMin:{v_min:.1f}, Max:{v_max:.1f}")
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    axes[0].axis('off')

    # 2. 0-255 Inferno (擬似カラー)
    im2 = axes[1].imshow(img_norm, cmap='inferno', vmin=0, vmax=1)
    axes[1].set_title("Luminance Heatmap (Inferno Colormap)")
    cbar2 = plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    cbar2.set_label('Normalized Intensity')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

## 5. 元画像(Raw)のROI統計量計算
最初のRaw画像（カラー変換済み）に対して、ROIを指定してRGB統計量を計算します。

In [ ]:
# ROI選択用の画像を表示（ターゲット画像を使用）
print("ROIを選択するためのウィンドウを開きます。ドラッグしてEnterキーで決定してください。")
# 表示用にビット深度を調整（16bitだとimshowで真っ黒になることがあるため8bit変換して表示）
disp_img = (rgb_target / 256).astype(np.uint8) if rgb_target.dtype == np.uint16 else rgb_target.copy()

try:
    r = cv2.selectROI("Select ROI", disp_img, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    
    # r = (x, y, w, h)
    x, y, w, h = r
    if w > 0 and h > 0:
        print(f"選択されたROI: x={x}, y={y}, w={w}, h={h}")
        
        # 統計量計算関数
        def calc_stats(name, image_rgb, x, y, w, h):
            roi = image_rgb[y:y+h, x:x+w]
            r_mean, r_std = np.mean(roi[:,:,0]), np.std(roi[:,:,0])
            g_mean, g_std = np.mean(roi[:,:,1]), np.std(roi[:,:,1])
            b_mean, b_std = np.mean(roi[:,:,2]), np.std(roi[:,:,2])
            
            print(f"--- {name} ---")
            print(f"ROI Mean: R={r_mean:.2f}, G={g_mean:.2f}, B={b_mean:.2f}")
            print(f"ROI Std : R={r_std:.2f},  G={g_std:.2f},  B={b_std:.2f}")

        # ターゲット画像とリファレンス画像の両方で計算
        calc_stats("Target Image (Raw RGB)", rgb_target, x, y, w, h)
        calc_stats("Reference Image (Raw RGB)", rgb_ref, x, y, w, h)
    else:
        print("ROIが選択されませんでした。")
except Exception as e:
    print(f"ROI選択中にエラーが発生しました: {e}")